# demo

> Isolated rendering of markup with current styles and behaviors, for notebooks and docs

In [1]:
#| default_exp demo

The registry holds styles and behaviors, and the behavior runtime turns them into JavaScript. To develop a component, we need to see the result and try its interactions. `demo` renders ordinary FastHTML markup in an isolated document with the current assets. The same preview is used both during notebook authoring and when viewing generated docs.


In [2]:
from fastcore.utils import *
from fastcore.meta import delegates
from fasthtml.common import to_xml, Div, Iframe, Button, Html, Head, Body, Style, Script
from fastcore.xml import highlight
from fasthtml.core import def_hdrs
from answerui.registry import reg, Hyperelement
import answerui.behaviors  # installs Registry.build_js
from fastlucide.icons import spritesheet

In [3]:
from answerui.registry import hypelt
from fastcore.test import *
from fasthtml.common import P, Details, Summary

## Rendering previews

A preview needs its own stylesheet and behavior script, the sprite sheet for icons, and htmx. FastHTML's `def_hdrs` gives the headers an app has by default: the charset and viewport metas, htmx 2, and FastHTML's helper script. With `htmx4`, as for `fast_app`, it gives htmx 4 instead, with the `htmx-config` meta that sets `metaCharacter` to `-`. We leave Surreal out.

htmx loads before the behavior script, which registers with it, and both are in the head before the page's content arrives. Supplying `css` replaces the registered stylesheet, which lets us experiment with styling independently of the rest of the library. The assets come from `reg` unless another registry is passed, so a registry of one's own previews the same way.

We make the preview's body size to its content, whatever an app's own `body` rule says. Otherwise, an app that makes its page fill the viewport would make the frame grow without end.

In [4]:
def demo_html(
    *fts,         # Markup to render
    registry=reg, # Registry whose stylesheet and behavior script the document carries
    css=None,     # Stylesheet for the document; None uses the registry's build_css()
    htmx4=False,  # Use htmx 4 with FastHTML's config, as `fast_app(htmx4=True)` does, instead of htmx 2
):
    "A standalone document containing markup, CSS, behaviors, the sprite sheet, and FastHTML's default headers"
    head = Head(*def_hdrs(htmx4=htmx4, surreal=False), Style(registry.build_css() if css is None else css), Script(registry.build_js()))
    return to_xml(Html(head, Body(*fts, spritesheet, style='min-block-size: auto')))


For rendering in notebooks, we use an iframe to isolate the preview, so the styling of the notebook (e.g. solveit's CSS) can't affect the rendered markup, and vice versa. Its `srcdoc` attribute holds the complete document from `demo_html`. By default, the iframe automatically resizes to fit its contents, and we include a replay button so that we can reload dynamic components (e.g. ones that auto-dismiss).

The frame holds the whole stylesheet and script, and a notebook stores what `demo` returns. So `demo` returns a `Preview` with two representations: the frame as HTML, for the page, and the markup alone as Markdown, for the notebook file and for a model reading a dialog. The frame's root carries `answerui-preview`, which `answerui.previews` uses to find frames in a saved notebook.

In [5]:
_sizer = Script(r"new ResizeObserver(() => frameElement.style.height = document.body.scrollHeight+24+'px').observe(document.body)")

class Preview:
    "A rendered preview: the frame as HTML for the page, the markup as Markdown for the notebook file and the model"
    def __init__(self, fts, frame): store_attr()
    def __repr__(self): return to_xml(self.fts)
    def _repr_mimebundle_(self, include=None, exclude=None): return {'text/html': to_xml(self.frame), 'text/markdown': highlight(self.fts)}, {'nbdev': {'strip': ['text/html']}}

def demo(
    *fts,         # Markup to render
    registry=reg, # Registry whose stylesheet and behavior script the preview carries
    css=None,     # Stylesheet for the document; None uses the registry's build_css()
    height=None,  # Fixed iframe height, e.g. '10rem', for top-layer content the frame can't measure; None follows the content
    width=None,   # Fixed iframe width, e.g. '24rem', to preview a narrow viewport; None fills the cell
    htmx4=False,  # Use htmx 4 with FastHTML's config, as `fast_app(htmx4=True)` does, instead of htmx 2
):
    "Render markup in an isolated, replayable notebook preview"
    doc = fts if height is not None else fts + (_sizer,)
    style = f'width:100%;box-sizing:border-box;border:1px solid #e3e3e8;border-radius:8px;height:{height or "auto"};'
    frame = Iframe(srcdoc=demo_html(*doc, registry=registry, css=css, htmx4=htmx4), title='answerui preview', style=style)
    replay = Button('↻', type='button', aria_label='replay', title='replay',
        onclick=r"const f=this.parentElement.nextElementSibling; f.srcdoc=f.srcdoc",
        style='border:0;background:none;font-size:14px;opacity:.6;cursor:pointer;padding:0 4px')
    return Preview(fts, Div(Div(replay, style='text-align:right'), frame, answerui_preview=True, style=f'width:{width or "100%"}'))


A preview does not require a registered component; Here `css=''` replaces the registry's stylesheet with an empty one, showing a native element exactly as the browser draws it:

In [6]:
demo(Details(Summary('More information'), P('This disclosure is a native details element.')), css='')

<details><summary>More information</summary>  <p>This disclosure is a native details element.</p>
</details>

Let's recreate the notice from the earlier notebooks. Its styles and behaviors register together. The dismiss button uses an ordinary click rule; `autodismiss` starts a timer when htmx loads the notice.

In [7]:
notice_css = r"""
notice-box {
  display: block;
  padding: .75rem 1rem;
  background: #f4f4f6;
  border: 1px solid #ccc;
  border-radius: 8px;
}
"""
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', on={
    ('& [dismiss]', 'click'): r"el.closest('notice-box').remove()",
    ('&[autodismiss]', 'init'): r"await new Promise(r => setTimeout(r, +el.getAttribute('autodismiss') || 4000)); el.remove()"},
    doc='A dismissible notice.')

Click Dismiss, then the ↻ icon to restore the notice:

In [8]:
demo(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)))

<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box>

What the notebook stores, and what a model reads from a dialog, is the preview's representations. The HTML is the frame, whose root carries `answerui-preview`. The Markdown is the markup alone. The metadata beside them marks the HTML with nbdev's `strip` flag, which tells `nbdev-clean` to drop the frame from the stored output and keep the Markdown:


In [9]:
p = demo(NoticeBox('Changes saved.'))
bundle, meta = p._repr_mimebundle_()
test_eq(set(bundle), {'text/html', 'text/markdown'})
assert 'answerui-preview' in bundle['text/html'] and 'srcdoc' in bundle['text/html']
assert 'srcdoc' not in bundle['text/markdown']
test_eq(meta, {'nbdev': {'strip': ['text/html']}})
print(bundle['text/markdown'])

```html
<notice-box>Changes saved.</notice-box>
```


This notice disappears after four seconds:

In [10]:
demo(NoticeBox('This notice disappears after four seconds.', autodismiss=4000), height='4rem')

<notice-box autodismiss="4000">This notice disappears after four seconds.</notice-box>

An app built with `fast_app(htmx4=True)` runs htmx 4 with FastHTML's config, and `htmx4` gives a preview the same headers. The notice's `init` rule then listens for `htmx-after-process`, the arrival event under that config, and the timer still starts:

In [11]:
h = demo_html(NoticeBox('x'), htmx4=True)
assert 'htmx.org@4' in h and 'metaCharacter' in h
demo(NoticeBox('This notice disappears after four seconds under htmx 4.', autodismiss=4000), htmx4=True, height='4rem')

<notice-box autodismiss="4000">This notice disappears after four seconds under htmx 4.</notice-box>

A custom stylesheet lets us work on a candidate without changing the registry. Here we reuse the notice's CSS but replace its grey background with a pale yellow. The registered behaviors still apply.

In [12]:
candidate_css = notice_css.replace('#f4f4f6', '#fff4cc')
demo(NoticeBox('Trying a warmer surface. ', Button('Dismiss', type='button', dismiss=True)), css=candidate_css)

<notice-box>Trying a warmer surface. <button type="button" dismiss>Dismiss</button></notice-box>

An application override takes a different path. Registering CSS in the `app` layer makes it part of subsequent previews and application pages. This rule leaves the notice's definition and behavior alone:

In [13]:
NoticeBox.rule('corners', 'border-radius: 1rem;', doc='Rounder notice corners.')
demo(NoticeBox('The current stylesheet includes the application override.'))


<notice-box>The current stylesheet includes the application override.</notice-box>

## Reusable examples

A component can provide an example for someone exploring the library. We attach a no-argument markup function to the hyperelement with `example`, which stores it as `NoticeBox.example`, and `NoticeBox.demo()` renders that markup with the component's registry.

Attaching another example replaces the previous one; several variants can be composed in one function.

In [14]:
def example(h):
    "Decorate a no-argument markup function as `h.example`, the markup `h.demo()` renders"
    def _attach(f):
        h.example = f
        return f
    return _attach

@patch
@delegates(demo, but=['registry'])
def demo(self:Hyperelement, **kwargs):
    "Render this hyperelement's example with its registry's current styles and behaviors"
    if not hasattr(self, 'example'): raise ValueError(f'{self.__name__} has no example; attach one with @example({self.__name__})')
    return demo(self.example(), registry=self.registry, **kwargs)


In [15]:
@example(NoticeBox)
def notice_example():
    return Div(NoticeBox('Changes saved. ', Button('Dismiss', type='button', dismiss=True)),
        NoticeBox('This notice stays visible.'), style='display:grid;gap:1rem')

The example itself is ordinary markup:

In [16]:
to_xml(NoticeBox.example())

'<div style="display:grid;gap:1rem">\n<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>\n'

Rendered with the current assets:

In [17]:
NoticeBox.demo()


<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

`NoticeBox.example()` is fresh markup, ordinary FastHTML that can be inspected or reused in a larger composition, and `NoticeBox.demo()` takes the preview options such as `height` and `css`. Each call builds a new document from the current example, styles, and behaviors. The replay button does something different: it reloads the preview's existing snapshot, restarting its interactions without consulting Python.

For a shipped component, the decorated example function is exported in its component notebook, so importing the module attaches the example without rendering anything.


A customization, then another preview. The rule is registered, so the new document carries it:

In [18]:
NoticeBox.rule('corners', 'border-radius: 0;', doc='Square notice corners.')
NoticeBox.demo()

<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

Redefining the component keeps the object, and its example with it:

In [19]:
NoticeBox = hypelt('NoticeBox', css=notice_css, layer='components', doc='A persistent notice.')
test_is(NoticeBox.example, notice_example)
NoticeBox.demo()

<div style="display:grid;gap:1rem">
<notice-box>Changes saved. <button type="button" dismiss>Dismiss</button></notice-box><notice-box>This notice stays visible.</notice-box></div>

## Export

In [20]:
#| hide
import nbdev; nbdev.nbdev_export()